In [ ]:
from pathlib import Path
from datetime import datetime

# === REQUIRED: set the folder you want to compress ===
SRC_DIR = Path("/lambda/nfs/objectdetection/object_detection_powerline_sleeve")  # e.g., r"C:\data\big_folder" or "/home/you/data"

# === OPTIONAL: set output zip path; if None, it will create next to SRC_DIR with a timestamp ===
DEST_ZIP = None  # or Path("/path/to/output/my_archive.zip")

# === OPTIONAL: exclude patterns (glob patterns relative to SRC_DIR) ===
EXCLUDES = [
    "*.tmp", "*.log", "*.DS_Store", "__pycache__/*", ".ipynb_checkpoints/*",
    "*.zip"
]

# === Compression options ===
COMPRESSION = "deflate"  # "deflate", "zstd" (Py 3.12+), "lzma", "bzip2", or "store"
COMPRESSLEVEL = 6        # typical: 5-6 for deflate; 3 for zstd is fast+good
FOLLOW_SYMLINKS = False

# === Rate tracking & logging ===
STATS_EVERY_SEC = 1.0     # how often to update/record stats
RATE_WINDOW_SEC = 15.0    # sliding window for instantaneous rate (seconds)
EMA_ALPHA = 0.2           # smoothing for EMA rate (0..1), higher = more reactive
LOG_CSV = None            # e.g., Path("zip_speed_log.csv") to save timeseries, or None to disable

In [ ]:
import os, sys, time, zipfile, math, csv
from pathlib import Path
from collections import deque

try:
    from tqdm import tqdm
except Exception:
    tqdm = None  # fallback: no progress bar

def _human_bytes(n: int) -> str:
    if n is None:
        return "?"
    units = ["B","KB","MB","GB","TB","PB"]
    if n == 0:
        return "0 B"
    k = int(math.floor(math.log(n, 1024)))
    k = min(k, len(units)-1)
    return f"{n / (1024**k):.2f} {units[k]}"

def _zipfile_params(kind: str):
    kind = (kind or "deflate").lower()
    if kind == "store":
        return zipfile.ZIP_STORED, None
    if kind == "deflate":
        return zipfile.ZIP_DEFLATED, "deflate"
    if kind == "bzip2":
        return zipfile.ZIP_BZIP2, "bzip2"
    if kind == "lzma":
        return zipfile.ZIP_LZMA, "lzma"
    if kind == "zstd" and hasattr(zipfile, "ZIP_ZSTD"):
        return zipfile.ZIP_ZSTD, "zstd"
    raise ValueError(f"Unknown or unsupported compression '{kind}' on this Python.")

def _is_subpath(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except Exception:
        return False

def _iter_files(src_dir: Path, excludes, follow_symlinks=False):
    excludes = excludes or []
    for root, _, files in os.walk(src_dir, followlinks=follow_symlinks):
        root_p = Path(root)
        for fn in files:
            fp = root_p / fn
            rel = fp.relative_to(src_dir)
            if any(rel.match(pat) for pat in excludes):
                continue
            yield fp, rel

def _calc_total_size(src_dir: Path, excludes, follow_symlinks=False):
    total = 0
    count = 0
    for fp, _ in _iter_files(src_dir, excludes, follow_symlinks):
        try:
            total += fp.stat().st_size
            count += 1
        except Exception:
            pass
    return total, count

def compress_folder_to_zip(
    src_dir: Path,
    dest_zip: Path | None = None,
    *,
    excludes=None,
    compression="deflate",
    compresslevel: int | None = None,
    follow_symlinks=False,
    stats_every_sec: float = 1.0,
    rate_window_sec: float = 10.0,
    ema_alpha: float = 0.2,
    log_csv: Path | None = None,
):
    MB = 1024 * 1024

    src_dir = Path(src_dir)
    if not src_dir.exists() or not src_dir.is_dir():
        raise FileNotFoundError(f"Source directory not found or not a directory: {src_dir}")

    if dest_zip is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        dest_zip = src_dir.with_name(f"{src_dir.name}_{ts}.zip")
    else:
        dest_zip = Path(dest_zip)

    if _is_subpath(dest_zip, src_dir):
        raise ValueError(f"Destination zip '{dest_zip}' is inside the source directory '{src_dir}'. "
                         "Choose an output path outside the source folder.")

    comp_const, comp_name = _zipfile_params(compression)
    zf_kwargs = {"compression": comp_const, "allowZip64": True}
    if compresslevel is not None:
        try:
            zf_kwargs["compresslevel"] = int(compresslevel)
        except Exception:
            pass

    total_bytes, total_files = _calc_total_size(src_dir, excludes, follow_symlinks)
    print(f"Zipping: {src_dir}")
    print(f"Output : {dest_zip}")
    print(f"Files  : {total_files:,}  |  Size: {_human_bytes(total_bytes)}")
    if comp_name:
        print(f"Method : {comp_name} (level={compresslevel})")
    else:
        print("Method : store (no compression)")

    dest_zip.parent.mkdir(parents=True, exist_ok=True)

    # Speed tracking state
    bytes_done = 0
    files_done = 0
    t0 = time.time()
    last_report = t0
    ema_rate_MBps = None

    # Sliding window for instantaneous rate
    window = deque()  # items: (timestamp, bytes_added)
    window_bytes = 0

    # Optional CSV logger
    csv_fh = None
    csv_writer = None
    if log_csv is not None:
        csv_fh = open(log_csv, "w", newline="", buffering=1)
        csv_writer = csv.writer(csv_fh)
        csv_writer.writerow([
            "timestamp_iso", "elapsed_s", "files_done", "bytes_done",
            "inst_MBps", "ema_MBps", "avg_MBps", "eta_s", "pct_complete"
        ])

    # Progress bar
    bar = tqdm(total=total_bytes or None, unit="B", unit_scale=True, desc="Compressing", leave=True) if tqdm else None

    def _report(now):
        nonlocal ema_rate_MBps, last_report
        elapsed = now - t0
        avg_MBps = (bytes_done / MB) / max(elapsed, 1e-6)

        # Prune old window entries
        while window and (now - window[0][0]) > rate_window_sec:
            _, b_old = window.popleft()
            nonlocal window_bytes
            window_bytes -= b_old

        # Instantaneous over window
        win_dt = (now - window[0][0]) if window else 0.0
        inst_MBps = (window_bytes / MB) / max(win_dt, 1e-6) if window and win_dt > 0 else 0.0

        # EMA
        if ema_rate_MBps is None:
            ema_rate_MBps = inst_MBps
        else:
            ema_rate_MBps = ema_alpha * inst_MBps + (1.0 - ema_alpha) * ema_rate_MBps

        # ETA using EMA if available; fallback to avg
        eff_MBps = ema_rate_MBps if ema_rate_MBps > 1e-6 else avg_MBps
        eta_s = ((total_bytes - bytes_done) / MB) / max(eff_MBps, 1e-6) if total_bytes else 0.0
        pct = (bytes_done / total_bytes * 100.0) if total_bytes else 0.0

        # Update progress bar postfix
        if bar:
            bar.set_postfix({
                "inst": f"{inst_MBps:5.1f} MB/s",
                "ema":  f"{ema_rate_MBps:5.1f} MB/s",
                "avg":  f"{avg_MBps:5.1f} MB/s",
                "eta":  f"{eta_s/60:6.1f} min"
            })
        else:
            # Fallback printing line (won't spam thanks to timing gate)
            print(f"[{pct:5.1f}%] inst={inst_MBps:5.1f} MB/s | "
                  f"ema={ema_rate_MBps:5.1f} MB/s | avg={avg_MBps:5.1f} MB/s | "
                  f"eta={eta_s/60:6.1f} min")

        # CSV log
        if csv_writer:
            csv_writer.writerow([
                datetime.fromtimestamp(now).isoformat(),
                f"{elapsed:.3f}",
                files_done,
                bytes_done,
                f"{inst_MBps:.3f}",
                f"{ema_rate_MBps:.3f}",
                f"{avg_MBps:.3f}",
                f"{eta_s:.3f}",
                f"{pct:.3f}",
            ])

        last_report = now

    try:
        with zipfile.ZipFile(dest_zip, mode="w", **zf_kwargs) as zf:
            for fp, rel in _iter_files(src_dir, excludes, follow_symlinks):
                size = 0
                try:
                    size = fp.stat().st_size
                    zf.write(fp, arcname=str(rel.as_posix()))
                except Exception as e:
                    # Use the bar (if present) to avoid breaking the progress line
                    msg = f"Skipped: {fp} ({e})"
                    (bar.write(msg) if bar else print(msg))
                finally:
                    files_done += 1
                    bytes_done += size

                    # Update window and progress
                    now = time.time()
                    window.append((now, size))
                    window_bytes += size
                    if bar:
                        bar.update(size)

                    # Periodic report
                    if (now - last_report) >= stats_every_sec:
                        _report(now)
        # Final report
        _report(time.time())
    finally:
        if bar:
            bar.close()
        if csv_fh:
            try: csv_fh.close()
            except: pass

    elapsed = time.time() - t0
    out_size = dest_zip.stat().st_size if dest_zip.exists() else 0
    ratio = (1 - (out_size / total_bytes)) * 100 if total_bytes else 0
    print(f"\nDone in {elapsed:.2f}s  |  Output size: {_human_bytes(out_size)}  "
          f"|  Compression: {ratio:.2f}%")

    return dest_zip

In [ ]:
# Resolve defaults and run
src = Path(SRC_DIR)
if DEST_ZIP is None:
    out = src.parent / f"{src.name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
else:
    out = Path(DEST_ZIP)

zip_path = compress_folder_to_zip(
    src,
    out,
    excludes=EXCLUDES,
    compression=COMPRESSION,
    compresslevel=COMPRESSLEVEL,
    follow_symlinks=FOLLOW_SYMLINKS,
    stats_every_sec=STATS_EVERY_SEC,
    rate_window_sec=RATE_WINDOW_SEC,
    ema_alpha=EMA_ALPHA,
    log_csv=LOG_CSV,
)

print(f"\nArchive created at: {zip_path}")

In [ ]:
import os, random, tempfile, time, zipfile
from pathlib import Path

SAMPLE_FILES = 2000
RANDOM_SEED = 42
COMPRESSION = "deflate"  # "deflate", "zstd" (Py3.12+), "lzma", "bzip2", or "store"
COMPRESSLEVEL = 6
SRC_DIR = Path("/lambda/nfs/objectdetection/object_detection_powerline_sleeve")
EXCLUDES = ["*.zip", "__pycache__/*", ".ipynb_checkpoints/*"]

def _comp_const(name):
    name = name.lower()
    if name == "store": return zipfile.ZIP_STORED
    if name == "deflate": return zipfile.ZIP_DEFLATED
    if name == "bzip2": return zipfile.ZIP_BZIP2
    if name == "lzma": return zipfile.ZIP_LZMA
    if name == "zstd" and hasattr(zipfile, "ZIP_ZSTD"): return zipfile.ZIP_ZSTD
    raise ValueError(f"Unsupported compression: {name}")

def pick_files(src: Path, n: int):
    files = []
    for root, _, fs in os.walk(src):
        for f in fs:
            p = Path(root) / f
            rel = p.relative_to(src)
            if any(rel.match(pat) for pat in EXCLUDES):
                continue
            files.append(p)
    random.Random(RANDOM_SEED).shuffle(files)
    return files[:min(n, len(files))], len(files)

def estimate_ratio(src: Path):
    files, total_count = pick_files(src, SAMPLE_FILES)
    if not files: 
        raise SystemExit("No files found in sample.")
    total_bytes = 0
    for root, _, fs in os.walk(src):
        for f in fs:
            p = Path(root) / f
            rel = p.relative_to(src)
            if any(rel.match(pat) for pat in EXCLUDES): continue
            try: total_bytes += p.stat().st_size
            except: pass

    tmp = Path(tempfile.gettempdir()) / f"__zip_ratio_{os.getpid()}.zip"
    if tmp.exists(): tmp.unlink()
    zkw = {"compression": _comp_const(COMPRESSION), "allowZip64": True}
    if COMPRESSLEVEL is not None: zkw["compresslevel"] = int(COMPRESSLEVEL)

    with zipfile.ZipFile(tmp, "w", **zkw) as zf:
        for p in files:
            zf.write(p, arcname=str(p.relative_to(src).as_posix()))
    sample_in = sum(p.stat().st_size for p in files)
    sample_out = tmp.stat().st_size
    try: tmp.unlink()
    except: pass

    ratio = sample_out / max(sample_in, 1)
    est_total = ratio * total_bytes
    return ratio, total_bytes, est_total

ratio, total_in, est_out = estimate_ratio(SRC_DIR)
print(f"Sample compression ratio: {ratio*100:.1f}% of original")
print(f"Estimated final size: {est_out/1024/1024/1024:.2f} GB out of {total_in/1024/1024/1024:.2f} GB")